In [1]:
import joblib

X_train = joblib.load(
    "X_train_strat.pkl"
)

X_test = joblib.load(
    "X_test_strat.pkl"
)

y_train = joblib.load(
    "y_train_strat.pkl"
)

y_test = joblib.load(
    "y_test_strat.pkl"
)

In [2]:
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_regression
)

In [3]:
preprocessor_compact = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.05)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=f_regression,
            k=500
        )
    )

])

In [4]:
from lightgbm import LGBMRegressor

from xgboost import XGBRegressor

from catboost import CatBoostRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.svm import SVR

from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.neural_network import MLPRegressor

In [5]:

# MODEL PIPELINES


lgbm_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        LGBMRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


xgb_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        XGBRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


cat_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        CatBoostRegressor(
            iterations=300,
            verbose=0,
            random_state=42,
            allow_writing_files=False
        )
    )

])


rf_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1, 
            max_depth=6
        )
    )

])

svr_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        SVR(
            kernel="rbf",
            C=10,
            epsilon=0.1
        )
    )

])

mlp_pipeline = Pipeline([
    
    ('preprocessor_compact', preprocessor_compact),
    (
        'model',MLPRegressor (
    
    
                hidden_layer_sizes = (256,128),
                activation = 'relu',
                solver = 'adam',
                alpha = 0.0001,
                batch_size = 'auto',
                learning_rate = 'adaptive',
                learning_rate_init = 0.0001,
                max_iter = 500,
                early_stopping= True,
                random_state = 42
        )
        
    )
    
])

et_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        ExtraTreesRegressor(

            n_estimators=300,

            random_state=42,

            n_jobs=-1,

            max_depth=10

        )
    )

])

hist_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        HistGradientBoostingRegressor(

            max_iter=300,

            learning_rate=0.05,

            max_depth=6,

            random_state=42

        )
    )

])



# STORE ALL MODELS


models = {

    "LightGBM": lgbm_pipeline,

    "XGBoost": xgb_pipeline,

    "CatBoost": cat_pipeline,

    "RandomForest": rf_pipeline,
    
    "SVR": svr_pipeline,
    
    "MLPRegressor": mlp_pipeline,
    
    "ExtraTreesRegressor": et_pipeline, 
    
    "HistGradientBoostingRegressor": hist_pipeline
    

}


models = {

    "LightGBM": lgbm_pipeline,

    "XGBoost": xgb_pipeline,

    "CatBoost": cat_pipeline,

    "RandomForest": rf_pipeline,

    "ExtraTrees": et_pipeline,

    "HistGradient": hist_pipeline,

    "SVR": svr_pipeline,

    "MLPRegressor": mlp_pipeline

}

In [6]:
from sklearn.model_selection import cross_val_score

results = []

for model_name, pipeline in models.items():

    cv_scores = cross_val_score(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring="r2",

        n_jobs=-1

    )

    results.append({

        "Model": model_name,

        "CV Mean R2": cv_scores.mean(),

        "CV Std": cv_scores.std()

    })

   
    print(model_name)
   

    print("CV Scores:")
    print(cv_scores)

    print("\nMean CV R2:")
    print(cv_scores.mean())

LightGBM
CV Scores:
[0.79241834 0.76078359 0.78376623 0.77899925 0.75947953]

Mean CV R2:
0.7750893894108231
XGBoost
CV Scores:
[0.76251608 0.73902286 0.76150292 0.74999396 0.73610241]

Mean CV R2:
0.7498276472854594
CatBoost
CV Scores:
[0.78914092 0.76182936 0.78164273 0.77666732 0.75952738]

Mean CV R2:
0.7737615438868526
RandomForest
CV Scores:
[0.72812357 0.70580921 0.72055951 0.72902579 0.70399394]

Mean CV R2:
0.7175024049473512
ExtraTrees
CV Scores:
[0.76017341 0.73292891 0.75707846 0.76448901 0.74417081]

Mean CV R2:
0.7517681209619095
HistGradient
CV Scores:
[0.79553479 0.76476871 0.77988411 0.78645406 0.75468211]

Mean CV R2:
0.7762647560525554
SVR
CV Scores:
[0.76266975 0.74817344 0.75876321 0.76397181 0.73999065]

Mean CV R2:
0.7547137737662155
MLPRegressor
CV Scores:
[0.74882685 0.73539102 0.75280968 0.75000353 0.73329764]

Mean CV R2:
0.7440657439212929


In [7]:
import pandas as pd

In [8]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(

    by="CV Mean R2",

    ascending=False

)

results_df

,Model,CV Mean R2,CV Std
5,HistGradient,0.776265,0.014739
0,LightGBM,0.775089,0.012955
2,CatBoost,0.773762,0.011420
6,SVR,0.754714,0.009219
4,ExtraTrees,0.751768,0.011604
1,XGBoost,0.749828,0.010977
7,MLPRegressor,0.744066,0.008070
3,RandomForest,0.717502,0.010716


In [9]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

In [10]:
import numpy as np

In [11]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

test_results = []

for model_name, pipeline in models.items():

    # TRAIN MODEL
  

    pipeline.fit(
        X_train,
        y_train
    )



    
    # TRAIN PREDICTIONS
    

    y_train_pred = pipeline.predict(
        X_train
    )



    
    # TEST PREDICTIONS
    

    y_test_pred = pipeline.predict(
        X_test
    )



    
    # TRAIN METRICS
    

    train_r2 = r2_score(
        y_train,
        y_train_pred
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            y_train_pred
        )
    )



    
    # TEST METRICS
    

    test_r2 = r2_score(
        y_test,
        y_test_pred
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred
        )
    )

    test_mae = mean_absolute_error(
        y_test,
        y_test_pred
    )



    
    # STORE RESULTS
    

    test_results.append({

        "Model": model_name,

        "Train_R2": train_r2,

        "Test_R2": test_r2,

        "Train_RMSE": train_rmse,

        "Test_RMSE": test_rmse,

        "Test_MAE": test_mae

    })



    
    # PRINT RESULTS
    

    print("\n")
    print(model_name)
    

    print("Train R2 :", train_r2)
    print("Test R2  :", test_r2)

    print("Train RMSE :", train_rmse)
    print("Test RMSE  :", test_rmse)

    print("Test MAE :", test_mae)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022346 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 115083
[LightGBM] [Info] Number of data points in the train set: 7984, number of used features: 500
[LightGBM] [Info] Start training from score -2.891766


c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(




LightGBM
Train R2 : 0.9466510858532475
Test R2  : 0.7874499864948796
Train RMSE : 0.5464107286020697
Test RMSE  : 1.095765866739421
Test MAE : 0.7190864122629151


XGBoost
Train R2 : 0.9707768072369536
Test R2  : 0.7673138312649432
Train RMSE : 0.40440848329602325
Test RMSE  : 1.146495841834712
Test MAE : 0.7622617888814152


CatBoost
Train R2 : 0.901540239691969
Test R2  : 0.7828477639373125
Train RMSE : 0.742310857621634
Test RMSE  : 1.107565329454706
Test MAE : 0.7467825259609818


RandomForest
Train R2 : 0.7658687378940097
Test R2  : 0.7227633746629799
Train RMSE : 1.1446853613957717
Test RMSE  : 1.251447114876536
Test MAE : 0.8910800864238339


ExtraTrees
Train R2 : 0.872265822555504
Test R2  : 0.7602822547074536
Train RMSE : 0.8454929363968727
Test RMSE  : 1.1636899714423985
Test MAE : 0.8068197680842427


HistGradient
Train R2 : 0.9009104206804947
Test R2  : 0.7839408002196446
Train RMSE : 0.7446812484311045
Test RMSE  : 1.1047743465959432
Test MAE : 0.7410773762289857


SVR
T

In [12]:
results_df = pd.DataFrame(
    test_results
)

results_df = results_df.sort_values(

    by="Test_R2",

    ascending=False

)

results_df

,Model,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Test_MAE
0,LightGBM,0.946651,0.787450,0.546411,1.095766,0.719086
5,HistGradient,0.900910,0.783941,0.744681,1.104774,0.741077
2,CatBoost,0.901540,0.782848,0.742311,1.107565,0.746783
7,MLPRegressor,0.842462,0.780275,0.938964,1.114108,0.766905
6,SVR,0.842588,0.772086,0.938589,1.134679,0.743577
1,XGBoost,0.970777,0.767314,0.404408,1.146496,0.762262
4,ExtraTrees,0.872266,0.760282,0.845493,1.163690,0.806820
3,RandomForest,0.765869,0.722763,1.144685,1.251447,0.891080
